In [1]:
import warnings
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.metrics import mean_squared_error
from scipy.stats import probplot
from ipywidgets import widgets
from IPython.display import display, clear_output
from datetime import datetime

#  Import Required Libraries

warnings.filterwarnings("ignore")

#  Configuration

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (14, 6)

# Define input widgets
stock1_input = widgets.Text(
    value='GBPAUD=X',
    placeholder='Enter first stock ticker',
    description='Stock 1:',
    disabled=False
)

stock2_input = widgets.Text(
    value='GBPJPY=X',
    placeholder='Enter second stock ticker',
    description='Stock 2:',
    disabled=False
)

start_date_picker = widgets.DatePicker(
    description='Start Date:',
    disabled=False,
    value=datetime(2020, 1, 1)
)

end_date_picker = widgets.DatePicker(
    description='End Date:',
    disabled=False,
    value=datetime(2024, 12, 31)
)

analyze_button = widgets.Button(
    description='Run Analysis',
    disabled=False,
    button_style='success', # 'success', 'info', 'warning', 'danger' or ''
    tooltip='Click to analyze',
    icon='play' # (FontAwesome icons)
)

# Output widget to display results
output_area = widgets.Output()

# Analysis function
def run_analysis(b):
    with output_area:
        clear_output()
        try:
            symbols = [stock1_input.value, stock2_input.value]
            start_date = start_date_picker.value.strftime('%Y-%m-%d')
            end_date = end_date_picker.value.strftime('%Y-%m-%d')

            print(f"Analyzing spread for {symbols[0]} and {symbols[1]} from {start_date} to {end_date}")

            # Download Data from Yahoo Finance
            data = yf.download(symbols, start=start_date, end=end_date)
            if data.empty:
                print("Could not fetch data for the given symbols and date range.")
                return

            # Select the 'Close' level from the MultiIndex columns
            data = data["Close"]
            data.dropna(inplace=True)

            if data.shape[1] < 2:
                print("Data could not be fetched for both symbols or there was no overlapping data.")
                return

            # 4. Calculate Spread
            data["spread"] = data[symbols[0]] - data[symbols[1]]
            spread = data["spread"]

            if spread.empty:
                print("Spread data is empty after calculating. Check symbol validity and date range.")
                return

            # Plot Spread
            plt.figure(figsize=(14, 6))
            plt.plot(spread, label=f"Spread: {symbols[0]} - {symbols[1]}")
            plt.title("Price Spread Between Assets")
            plt.xlabel("Date")
            plt.ylabel("Spread")
            plt.legend()
            plt.grid()
            plt.tight_layout()
            plt.show()

            #  Plot ACF and PACF
            plt.figure(figsize=(14, 6))
            plt.subplot(121)
            plot_acf(spread, lags=30, ax=plt.gca())
            plt.title("Autocorrelation of Spread")

            plt.subplot(122)
            plot_pacf(spread, lags=30, ax=plt.gca())
            plt.title("Partial Autocorrelation of Spread")
            plt.tight_layout()
            plt.show()

            #  Fit ARIMA Model

            # Convert spread to a NumPy array
            spread_array = spread.values

            # Check for non-numeric values or NaNs in spread_array
            if np.isnan(spread_array).any():
                 print("Warning: NaNs found in the spread data after dropping. ARIMA model might fail.")

            # Choose ARIMA order based on ACF/PACF or AIC (manual selection here)
            # You might want to add a mechanism to automatically find the best order
            arima_order = (1, 0, 1)  # ARIMA(p,d,q)

            try:
                model = ARIMA(spread_array, order=arima_order)
                model_fit = model.fit()

                # Display model summary
                print(model_fit.summary())

                # 9. Diagnostic Plots (Residuals)
                residuals = model_fit.resid

                fig, axes = plt.subplots(2, 2, figsize=(14, 8))

                # Residual time series
                axes[0, 0].plot(residuals)
                axes[0, 0].set_title("Residuals Over Time")

                # Q-Q plot
                probplot(residuals, dist="norm", plot=axes[0, 1])
                axes[0, 1].set_title("Normal Q-Q Plot")

                # ACF
                plot_acf(residuals, lags=30, ax=axes[1, 0])
                axes[1, 0].set_title("ACF of Residuals")

                # PACF
                plot_pacf(residuals, lags=30, ax=axes[1, 1])
                axes[1, 1].set_title("PACF of Residuals")

                plt.tight_layout()
                plt.show()

                # 10. Forecast Spread
                forecast_steps = 20
                forecast_result = model_fit.get_forecast(steps=forecast_steps)
                forecast_mean = forecast_result.predicted_mean
                conf_int = forecast_result.conf_int()

                # 11. Plot Forecast
                plt.figure(figsize=(14, 6))
                # Plot last 100 historical points for better visualization
                plt.plot(spread.index[-100:], spread[-100:], label="Historical Spread")

                # Create forecast index starting from the date after the last historical date
                if not spread.empty:
                    last_historical_date = spread.index[-1]
                    forecast_index = pd.date_range(start=last_historical_date, periods=forecast_steps + 1, freq=spread.index.freq)[1:]
                else:
                     print("Cannot generate forecast index as historical spread is empty.")
                     return


                plt.plot(forecast_index, forecast_mean, color="red", label="Forecast")
                plt.fill_between(
                    forecast_index,
                    conf_int[:, 0],
                    conf_int[:, 1],
                    color="pink",
                    alpha=0.3,
                    label="95% Confidence Interval"
                )
                plt.title("ARIMA Forecast of Spread")
                plt.xlabel("Date")
                plt.ylabel("Spread")
                plt.legend()
                plt.grid()
                plt.tight_layout()
                plt.show()

            except Exception as arima_e:
                print(f"Error fitting ARIMA model or forecasting: {arima_e}")
                print("This might happen if the data is too short or has issues.")


        except Exception as e:
            print(f"An error occurred: {e}")

# Link the button click event to the function
analyze_button.on_click(run_analysis)

# Display the widgets
display(widgets.VBox([
    widgets.HBox([stock1_input, stock2_input]),
    widgets.HBox([start_date_picker, end_date_picker]),
    analyze_button,
    output_area
]))

